In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns


## 1. Load Feature-Engineered Dataset

The dataset used in this notebook contains cleaned and feature-engineered variables from Milestone 2.


In [2]:
df = pd.read_csv("../data/processed/visa_feature_engineered.csv")
df.shape, df.columns


((120000, 25),
 Index(['age', 'nationality', 'current_residence', 'marital_status',
        'dependents', 'education_level', 'occupation', 'work_experience_years',
        'sponsorship_status', 'financial_status', 'language_proficiency',
        'purpose_of_travel', 'intended_duration', 'travel_history',
        'target_country', 'visa_category (Label)', 'previous_visa_rejections',
        'criminal_record_check', 'medical_exam_status',
        'document_completeness_score', 'visa_fee_payment_status',
        'processing_time_days', 'seasonal_index', 'country_avg_processing_time',
        'risk_score'],
       dtype='object'))

## 2. Define Features and Target Variable

The target variable is visa processing time in days.
All remaining columns are used as independent features.


In [3]:
y = df['processing_time_days']
X = df.drop(columns=['processing_time_days'])


## 3. Train–Test Split

The dataset is split into 70% training and 30% testing to evaluate model generalization.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


In [5]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)


## 4. Model Training

Two regression models are trained:
- Linear Regression (baseline)
- Random Forest Regressor


In [6]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)


In [7]:
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)


## 5. Model Evaluation Metrics

Models are evaluated using MAE, RMSE, and R² score.


In [8]:
def evaluate_model(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

lr_metrics = evaluate_model(y_test, y_pred_lr)
rf_metrics = evaluate_model(y_test, y_pred_rf)


## 6. Model Comparison


In [9]:
comparison_df = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [lr_metrics["MAE"], rf_metrics["MAE"]],
    "RMSE": [lr_metrics["RMSE"], rf_metrics["RMSE"]],
    "R2 Score": [lr_metrics["R2"], rf_metrics["R2"]]
})

comparison_df


,Model,MAE,RMSE,R2 Score
0,Linear Regression,2.412878,3.026524,0.957054
1,Random Forest,2.470466,3.097027,0.955030


## 7. Model Selection and Fine-Tuning

The best-performing model is selected based on evaluation metrics and fine-tuned to improve performance.


In [10]:
param_grid = {
    'n_estimators': [50],
    'max_depth': [10, 15],
    'min_samples_split': [5]
}

grid = GridSearchCV(
    RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),
    param_grid,
    cv=3,
    scoring='neg_mean_absolute_error'
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_


## 8. Final Model Evaluation


In [11]:
y_pred_best = best_model.predict(X_test)
best_metrics = evaluate_model(y_test, y_pred_best)
best_metrics


{'MAE': 2.4205911827684314,
 'RMSE': 3.036786922128021,
 'R2': 0.9567625373795732}

In [12]:
tuned_rf_metrics = best_metrics

final_comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Tuned Random Forest"],
    "MAE": [
        lr_metrics["MAE"],
        rf_metrics["MAE"],
        tuned_rf_metrics["MAE"]
    ],
    "RMSE": [
        lr_metrics["RMSE"],
        rf_metrics["RMSE"],
        tuned_rf_metrics["RMSE"]
    ],
    "R2 Score": [
        lr_metrics["R2"],
        rf_metrics["R2"],
        tuned_rf_metrics["R2"]
    ]
})

final_comparison


,Model,MAE,RMSE,R2 Score
0,Linear Regression,2.412878,3.026524,0.957054
1,Random Forest,2.470466,3.097027,0.955030
2,Tuned Random Forest,2.420591,3.036787,0.956763


In [13]:
comparison_df_sorted = final_comparison.sort_values(
    by=["MAE", "RMSE", "R2 Score"],
    ascending=[True, True, False]
)

comparison_df_sorted


,Model,MAE,RMSE,R2 Score
0,Linear Regression,2.412878,3.026524,0.957054
2,Tuned Random Forest,2.420591,3.036787,0.956763
1,Random Forest,2.470466,3.097027,0.955030


In [14]:
best_model_name = comparison_df_sorted.iloc[0]["Model"]
print("Final model selected:", best_model_name)

if best_model_name == "Linear Regression":
    final_model = lr
elif best_model_name == "Random Forest":
    final_model = rf
else:
    final_model = best_model  # tuned RF


Final model selected: Linear Regression


In [15]:
y_pred_final = final_model.predict(X_test)
evaluate_model(y_test, y_pred_final)


{'MAE': 2.4128775861528187,
 'RMSE': 3.0265240597615413,
 'R2': 0.9570542867360047}

## 9. Save Trained Model


In [16]:
import joblib
joblib.dump(final_model, "visa_processing_time_model.pkl")


['visa_processing_time_model.pkl']